# 1.Load and Extract Data

In [25]:
import os
os.listdir()

['Spam SMS Detection.ipynb', 'spam.csv']

In [26]:
import pandas as pd
data=pd.read_csv("spam.csv",encoding='ISO-8859-1')
data.rename(columns={
    'v1':'Status',
    'v2':'Text1',
    'Unnamed: 2':'Text2',
    'Unnamed: 3':'Text3',
    'Unnamed: 4':'Text4'},inplace=True)
data

,Status,Text1,Text2,Text3,Text4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


# 2.Exploratory Data Analysis(EDA)

In [27]:
data.isnull().sum()

Status       0
Text1        0
Text2     5522
Text3     5560
Text4     5566
dtype: int64

In [28]:
data['Text']=data['Text1'].fillna(" ").astype(str)+" "+data['Text2'].fillna(" ").astype(str)+" "+data['Text3'].fillna(" ").astype(str)+" "+data['Text4'].fillna(" ")
data=data.drop(columns=['Text1','Text2','Text3','Text4'])
data

,Status,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [29]:
data.isnull().sum()

Status    0
Text      0
dtype: int64

In [30]:
data.info()


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Status  5572 non-null   str  
 1   Text    5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


# 3.Feature Extraction

In [31]:
from sklearn.preprocessing import LabelEncoder
lr=LabelEncoder()
data['Status']=lr.fit_transform(data['Status'])
data

,Status,Text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [32]:
x=data['Text']
y=data['Status']

# 4.Model Building by Fit the TF-IDF vectorizer

In [33]:
from sklearn.model_selection import train_test_split
x_train, x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)




(4457,)
(1115,)
(4457,)
(1115,)


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(stop_words='english',max_features=5000)
x_train_tfidf=vectorizer.fit_transform(x_train)
x_test_tfidf=vectorizer.transform(x_test)

# 5.Model building & Evaluation - Logistic Regression

In [35]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression()
lr_model.fit(x_train_tfidf,y_train)
y_pre_lr=lr_model.predict(x_test_tfidf)

In [36]:
from sklearn.metrics import classification_report,accuracy_score
print("Logistic Regression Classifier:")
print(classification_report(y_test,y_pre_lr))
print("Accyracy:",accuracy_score(y_test,y_pre_lr))

Logistic Regression Classifier:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       0.97      0.72      0.83       150

    accuracy                           0.96      1115
   macro avg       0.97      0.86      0.90      1115
weighted avg       0.96      0.96      0.96      1115

Accyracy: 0.9596412556053812


# Model Testing

In [38]:
new_messages=[
    "Hey!how are you?",
    "You've been selected for a vacation trip free! Click here to claim your free Tickets",
    "Iam at classroom , I call back you later.",
    "Congratulations! You've won a prize for $2000. Call now to claim your prize"

]

new_messages_tfidf=vectorizer.transform(new_messages)

predictions_lr=lr_model.predict(new_messages_tfidf)

l_map={0:'ham',1:'spam'}
for i,message in enumerate(new_messages):
    print(f"Message:{message}")
    print(f"Status :\tLogistic Regression Prediction   :{l_map[predictions_lr[i]]}")
    print("="*100)



Message:Hey!how are you?
Status :	Logistic Regression Prediction   :ham
Message:You've been selected for a vacation trip free! Click here to claim your free Tickets
Status :	Logistic Regression Prediction   :spam
Message:Iam at classroom , I call back you later.
Status :	Logistic Regression Prediction   :ham
Message:Congratulations! You've won a prize for $2000. Call now to claim your prize
Status :	Logistic Regression Prediction   :spam
